# LLMOps Practice Notebook: Evaluation & Monitoring

This notebook demonstrates how to evaluate a RAG pipeline and log custom metrics to Application Insights.

## 1. Setup & Evaluation (using RAGAS)
Evaluation is the core of CI/CD for LLMs.

In [ ]:
# !pip install ragas
import os
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_precision
from datasets import Dataset

# Mock data representing a RAG response
data_samples = {
    'question': ['How do I reset my password?', 'What is the vacation policy?'],
    'answer': [
        'Go to settings and click reset.', 
        'Full-time employees get 20 days per year.'
    ],
    'contexts' : [
        ['Setting page has a password reset button.'], 
        ['The policy manual states 20 days of vacation for FTEs.']
    ],
    'ground_truth': [
        'Go to settings and click the password reset button.', 
        'Full-time employees are entitled to 20 days of annual leave.'
    ]
}

dataset = Dataset.from_dict(data_samples)

# In a real scenario, you would pass your Azure OpenAI credentials here
# score = evaluate(dataset, metrics=[faithfulness, answer_relevance, context_precision])
# print(score)

## 2. Monitoring with Application Insights
Log custom metrics for tokens and latency.

In [ ]:
from azure.monitor.opentelemetry import configure_azure_monitor
from opentelemetry import metrics
import time

# Configure Azure Monitor (requires CONNECTION_STRING)
# configure_azure_monitor(connection_string="InstrumentationKey=...")

meter = metrics.get_meter("llm_observability")

# Define metrics
token_counter = meter.create_counter("llm_tokens", unit="1", description="Total tokens used")
latency_histogram = meter.create_histogram("llm_latency", unit="ms", description="LLM response latency")

def log_llm_call(tokens, duration_ms, model_name):
    attributes = {"model": model_name}
    token_counter.add(tokens, attributes)
    latency_histogram.record(duration_ms, attributes)
    print(f"Logged {tokens} tokens and {duration_ms}ms latency for {model_name}")

log_llm_call(150, 2400, "gpt-4o")

## 3. PII Redaction Practice
Simple regex-based redaction (in production, use Presidio).

In [ ]:
import re

def redact_pii(text):
    # Redact email
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL_REDACTED]', text)
    # Redact phone (simple)
    text = re.sub(r'\b\d{10}\b', '[PHONE_REDACTED]', text)
    return text

raw_text = "My email is john.doe@example.com and phone is 1234567890."
print(redact_pii(raw_text))